# 04 风险分层策略

本文件承接 `03` 的风险因子分析，把单变量风险信号组合成可解释的风险分层规则。

本阶段不是建模，而是做一套可以被业务理解、复核和落地的规则分层。


## 1. 本阶段目标

本阶段回答三个问题：

1. 哪些客户应划为高风险？
2. 每个风险层有多少客户，坏客户率是多少？
3. 如果对高风险层采取拒绝或人工复核，会影响多少客户、捕获多少坏客户？

核心原则：规则要简单、可解释、可审计。


## 2. 导入工具和读取数据


In [3]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", font="Microsoft YaHei")

PROJECT_ROOT = Path(r"E:\DataAnalysis\DataAnalysisRoad\Projects\01_Credit_Risk_Analysis")
CLEAN_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "credit_risk_cleaned.csv"
OUTPUT_TABLE_DIR = PROJECT_ROOT / "output" / "tables"
OUTPUT_FIGURE_DIR = PROJECT_ROOT / "output" / "figures"

OUTPUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_DATA_PATH


WindowsPath('E:/DataAnalysis/DataAnalysisRoad/Projects/01_Credit_Risk_Analysis/data/processed/credit_risk_cleaned.csv')

In [4]:
df = pd.read_csv(CLEAN_DATA_PATH)

sample_count = len(df)
bad_count = df["is_bad_customer"].sum()
overall_bad_rate = df["is_bad_customer"].mean()

pd.DataFrame({
    "指标": ["样本数", "坏客户数", "整体坏客户率"],
    "结果": [sample_count, bad_count, overall_bad_rate],
})


,指标,结果
0,样本数,"150,000.0000"
1,坏客户数,"10,026.0000"
2,整体坏客户率,0.0668


## 3. 定义风险信号和分层标准

先定义量化标准，再给风险信号分级，不能凭直觉划分强弱。

本项目采用以下标准：

| 信号等级 | 标准 | 用途 |
|---|---|---|
| 直接高风险触发信号 | 相对整体坏客户率 >= 5，且命中人数 >= 500 | 单独命中即可进入高风险 |
| 重点叠加信号 | 3 <= 相对整体坏客户率 < 5，且命中人数 >= 500 | 多个叠加后进入高风险 |
| 普通叠加信号 | 1.5 <= 相对整体坏客户率 < 3，且命中人数 >= 500 | 辅助区分中高风险 |
| 观察信号 | 命中人数 < 500，或业务含义不够稳定 | 暂不参与正式分层，只做监控观察 |

说明：

- `DPD30+` 是重点叠加信号，不是直接高风险触发信号；
- `open_credit_count = 0` 的相对风险高于 `DPD30+`，也应归为重点叠加信号；
- 样本过小的信号即使坏客户率较高，也不能贸然进入正式规则；
- `open_credit_count > 40` 和 `dependent_count > 5` 都属于样本量较小的观察信号。


In [8]:
strategy_df = df.copy()

# 直接高风险触发信号：相对整体坏客户率 >= 5，且样本量足够
strategy_df["risk_90dpd_2plus"] = strategy_df["past_due_90_count"].ge(2).astype(int)
strategy_df["risk_90dpd_1"] = strategy_df["past_due_90_count"].eq(1).astype(int)
strategy_df["risk_util_100_1000"] = strategy_df["credit_utilization"].between(1, 10, inclusive="right").astype(int)

# 重点叠加信号：相对整体坏客户率在 3 到 5 倍之间，且样本量足够
strategy_df["risk_dpd30_plus"] = strategy_df["ever_dpd30_plus"].eq(1).astype(int)
strategy_df["risk_open_credit_zero"] = strategy_df["open_credit_count"].eq(0).astype(int)

# 普通叠加信号：相对整体坏客户率在 1.5 到 3 倍之间，且样本量足够
strategy_df["risk_util_70_100"] = strategy_df["credit_utilization"].between(0.7, 1, inclusive="right").astype(int)
strategy_df["risk_debt_100_500"] = strategy_df["debt_ratio"].between(1, 5, inclusive="right").astype(int)
strategy_df["risk_young_35_under"] = strategy_df["age"].le(35).astype(int)

# 观察信号：样本量过小，暂不参与正式分层
strategy_df["risk_dependent_5plus"] = strategy_df["dependent_count"].gt(5).astype(int)
strategy_df["risk_open_credit_over_40"] = strategy_df["open_credit_count"].gt(40).astype(int)
#strategy_df.head()
direct_high_signal_cols = [
    "risk_90dpd_2plus",
    "risk_90dpd_1",
    "risk_util_100_1000",
]

key_additive_signal_cols = [
    "risk_dpd30_plus",
    "risk_open_credit_zero",
]

standard_additive_signal_cols = [
    "risk_util_70_100",
    "risk_debt_100_500",
    "risk_young_35_under",
]

observation_signal_cols = [
    "risk_dependent_5plus",
    "risk_open_credit_over_40",
]

segmentation_signal_cols = direct_high_signal_cols + key_additive_signal_cols + standard_additive_signal_cols
all_signal_cols = segmentation_signal_cols + observation_signal_cols

strategy_df["direct_high_signal_count"] = strategy_df[direct_high_signal_cols].sum(axis=1)
strategy_df["key_additive_signal_count"] = strategy_df[key_additive_signal_cols].sum(axis=1)
strategy_df["standard_additive_signal_count"] = strategy_df[standard_additive_signal_cols].sum(axis=1)
strategy_df["risk_signal_count"] = strategy_df[segmentation_signal_cols].sum(axis=1)

strategy_df[[
    "customer_id",
    "is_bad_customer",
    "direct_high_signal_count",
    "key_additive_signal_count",
    "standard_additive_signal_count",
    "risk_signal_count",
]].head()


,customer_id,is_bad_customer,direct_high_signal_count,key_additive_signal_count,standard_additive_signal_count,risk_signal_count
0,1,1,0,1,1,2
1,2,0,0,0,1,1
2,3,0,1,1,0,2
3,4,0,0,0,1,1
4,5,0,0,1,1,2


## 4. 单个风险信号表现

检查每个风险信号的覆盖人数、坏客户率和相对整体坏客户率。

这里同时展示“信号等级”和“是否参与分层”，用于审计规则设定是否符合标准。


In [9]:
risk_signal_names = {
    "risk_90dpd_2plus": "90天以上逾期2次及以上",
    "risk_90dpd_1": "90天以上逾期1次",
    "risk_util_100_1000": "额度使用率100%-1000%",
    "risk_dpd30_plus": "曾经DPD30+",
    "risk_open_credit_zero": "开放信贷账户数为0",
    "risk_util_70_100": "额度使用率70%-100%",
    "risk_debt_100_500": "负债率100%-500%",
    "risk_young_35_under": "35岁及以下",
    "risk_dependent_5plus": "家属数量5人以上",
    "risk_open_credit_over_40": "开放信贷账户数大于40",
}

risk_signal_level = {
    "risk_90dpd_2plus": "直接高风险触发信号",
    "risk_90dpd_1": "直接高风险触发信号",
    "risk_util_100_1000": "直接高风险触发信号",
    "risk_dpd30_plus": "重点叠加信号",
    "risk_open_credit_zero": "重点叠加信号",
    "risk_util_70_100": "普通叠加信号",
    "risk_debt_100_500": "普通叠加信号",
    "risk_young_35_under": "普通叠加信号",
    "risk_dependent_5plus": "观察信号",
    "risk_open_credit_over_40": "观察信号",
}

signal_rows = []

for col in all_signal_cols:
    mask = strategy_df[col].eq(1)
    bad_rate = strategy_df.loc[mask, "is_bad_customer"].mean()
    signal_rows.append({
        "风险信号": risk_signal_names[col],
        "字段": col,
        "信号等级": risk_signal_level[col],
        "是否参与分层": "否" if col in observation_signal_cols else "是",
        "命中人数": mask.sum(),
        "命中占比": mask.mean(),
        "命中坏客户率": bad_rate,
        "相对整体坏客户率": bad_rate / overall_bad_rate,
    })

risk_signal_summary = pd.DataFrame(signal_rows).sort_values("相对整体坏客户率", ascending=False)
risk_signal_summary


,风险信号,字段,信号等级,是否参与分层,命中人数,命中占比,命中坏客户率,相对整体坏客户率
0,90天以上逾期2次及以上,risk_90dpd_2plus,直接高风险触发信号,是,2826,0.0188,0.5520,8.2588
2,额度使用率100%-1000%,risk_util_100_1000,直接高风险触发信号,是,3080,0.0205,0.3961,5.9262
1,90天以上逾期1次,risk_90dpd_1,直接高风险触发信号,是,5243,0.0350,0.3366,5.0365
4,开放信贷账户数为0,risk_open_credit_zero,重点叠加信号,是,1888,0.0126,0.2564,3.8354
3,曾经DPD30+,risk_dpd30_plus,重点叠加信号,是,30094,0.2006,0.2198,3.2886
5,额度使用率70%-100%,risk_util_70_100,普通叠加信号,是,26627,0.1775,0.1772,2.6504
9,开放信贷账户数大于40,risk_open_credit_over_40,观察信号,否,62,0.0004,0.1290,1.9305
8,家属数量5人以上,risk_dependent_5plus,观察信号,否,245,0.0016,0.1265,1.8930
6,负债率100%-500%,risk_debt_100_500,普通叠加信号,是,5491,0.0366,0.1182,1.7683
7,35岁及以下,risk_young_35_under,普通叠加信号,是,21485,0.1432,0.1113,1.6650


## 5. 设计风险分层规则

基于上面的量化标准，采用三层规则：

- 高风险：命中任一直接高风险触发信号；或命中 2 个及以上重点叠加信号；或命中 1 个重点叠加信号且 2 个及以上普通叠加信号；
- 中风险：未进入高风险，但命中任一重点叠加信号或普通叠加信号；
- 低风险：未命中任何参与分层的风险信号。

观察信号不参与正式分层，只保留在信号表现表中观察。


In [10]:
high_risk_rule = (
    strategy_df["direct_high_signal_count"].ge(1)
    | strategy_df["key_additive_signal_count"].ge(2)
    | (
        strategy_df["key_additive_signal_count"].ge(1)
        & strategy_df["standard_additive_signal_count"].ge(2)
    )
)

medium_risk_rule = (~high_risk_rule) & strategy_df["risk_signal_count"].ge(1)
low_risk_rule = (~high_risk_rule) & (~medium_risk_rule)

strategy_df["risk_segment"] = "低风险"
strategy_df.loc[medium_risk_rule, "risk_segment"] = "中风险"
strategy_df.loc[high_risk_rule, "risk_segment"] = "高风险"

strategy_df[[
    "customer_id",
    "is_bad_customer",
    "direct_high_signal_count",
    "key_additive_signal_count",
    "standard_additive_signal_count",
    "risk_signal_count",
    "risk_segment",
]].head()


,customer_id,is_bad_customer,direct_high_signal_count,key_additive_signal_count,standard_additive_signal_count,risk_signal_count,risk_segment
0,1,1,0,1,1,2,中风险
1,2,0,0,0,1,1,中风险
2,3,0,1,1,0,2,高风险
3,4,0,0,0,1,1,中风险
4,5,0,0,1,1,2,中风险


## 6. 风险分层效果评估

分层效果重点看：

- 每层客户数和客户占比；
- 每层坏客户率；
- 每层捕获了多少坏客户；
- 坏客户捕获率。


In [11]:
segment_order = ["高风险", "中风险", "低风险"]

segment_summary = strategy_df.groupby("risk_segment")["is_bad_customer"].agg(["count", "sum", "mean"]).reindex(segment_order).reset_index()
segment_summary.columns = ["风险层级", "客户数", "坏客户数", "坏客户率"]
segment_summary["客户占比"] = segment_summary["客户数"] / sample_count
segment_summary["坏客户捕获率"] = segment_summary["坏客户数"] / bad_count
segment_summary["相对整体坏客户率"] = segment_summary["坏客户率"] / overall_bad_rate

segment_summary


,风险层级,客户数,坏客户数,坏客户率,客户占比,坏客户捕获率,相对整体坏客户率
0,高风险,12385,4510,0.3642,0.0826,0.4498,5.4481
1,中风险,49586,4154,0.0838,0.3306,0.4143,1.2533
2,低风险,88029,1362,0.0155,0.5869,0.1358,0.2315


In [12]:
plt.figure(figsize=(7, 4))
sns.barplot(data=segment_summary, x="风险层级", y="坏客户率", order=segment_order, color="#4C78A8")
plt.axhline(overall_bad_rate, color="red", linestyle="--", label="整体坏客户率")
plt.title("风险分层坏客户率")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_FIGURE_DIR / "risk_segment_bad_rate.png", dpi=150)
plt.close()


## 7. 策略模拟：高风险拒绝或人工复核

这里不直接建议拒绝，只做影响评估。

常见策略含义：

- 拒绝高风险：降低坏客户，但会误伤一部分好客户；
- 人工复核高风险：保留业务机会，但增加审核成本。


In [13]:
high_risk_mask = strategy_df["risk_segment"].eq("高风险")

strategy_impact = pd.DataFrame({
    "策略": ["高风险拒绝/复核"],
    "影响客户数": [high_risk_mask.sum()],
    "影响客户占比": [high_risk_mask.mean()],
    "捕获坏客户数": [strategy_df.loc[high_risk_mask, "is_bad_customer"].sum()],
    "坏客户捕获率": [strategy_df.loc[high_risk_mask, "is_bad_customer"].sum() / bad_count],
    "误伤好客户数": [(high_risk_mask.sum() - strategy_df.loc[high_risk_mask, "is_bad_customer"].sum())],
    "高风险层坏客户率": [strategy_df.loc[high_risk_mask, "is_bad_customer"].mean()],
})

strategy_impact


,策略,影响客户数,影响客户占比,捕获坏客户数,坏客户捕获率,误伤好客户数,高风险层坏客户率
0,高风险拒绝/复核,12385,0.0826,4510,0.4498,7875,0.3642


## 8. 高风险客户命中原因拆解

这张表用于解释高风险客户为什么被划入高风险。

只统计参与正式分层的信号，不统计观察信号。


In [14]:
high_risk_reason_summary = []

for col in segmentation_signal_cols:
    mask = high_risk_mask & strategy_df[col].eq(1)
    high_risk_reason_summary.append({
        "高风险命中原因": risk_signal_names[col],
        "信号等级": risk_signal_level[col],
        "命中高风险客户数": mask.sum(),
        "占高风险客户比例": mask.sum() / high_risk_mask.sum(),
    })

high_risk_reason_summary = pd.DataFrame(high_risk_reason_summary).sort_values("命中高风险客户数", ascending=False)
high_risk_reason_summary


,高风险命中原因,信号等级,命中高风险客户数,占高风险客户比例
3,曾经DPD30+,重点叠加信号,10890,0.8793
5,额度使用率70%-100%,普通叠加信号,6454,0.5211
1,90天以上逾期1次,直接高风险触发信号,5243,0.4233
7,35岁及以下,普通叠加信号,4442,0.3587
2,额度使用率100%-1000%,直接高风险触发信号,3080,0.2487
0,90天以上逾期2次及以上,直接高风险触发信号,2826,0.2282
4,开放信贷账户数为0,重点叠加信号,1166,0.0941
6,负债率100%-500%,普通叠加信号,781,0.0631


## 9. 保存风险分层结果

保存三类结果：

- 客户级分层结果；
- 分层汇总和策略影响表；
- 风险信号分级审计表。


In [15]:
segmented_data_path = OUTPUT_TABLE_DIR / "customer_risk_segments.csv"
segment_summary_path = OUTPUT_TABLE_DIR / "risk_segment_summary.csv"
strategy_impact_path = OUTPUT_TABLE_DIR / "strategy_impact_high_risk.csv"
high_risk_reason_path = OUTPUT_TABLE_DIR / "high_risk_reason_summary.csv"
risk_signal_summary_path = OUTPUT_TABLE_DIR / "risk_signal_summary.csv"

strategy_df.to_csv(segmented_data_path, index=False, encoding="utf-8-sig")
segment_summary.to_csv(segment_summary_path, index=False, encoding="utf-8-sig")
strategy_impact.to_csv(strategy_impact_path, index=False, encoding="utf-8-sig")
high_risk_reason_summary.to_csv(high_risk_reason_path, index=False, encoding="utf-8-sig")
risk_signal_summary.to_csv(risk_signal_summary_path, index=False, encoding="utf-8-sig")

segmented_data_path, segment_summary_path, strategy_impact_path, high_risk_reason_path, risk_signal_summary_path


(WindowsPath('E:/DataAnalysis/DataAnalysisRoad/Projects/01_Credit_Risk_Analysis/output/tables/customer_risk_segments.csv'),
 WindowsPath('E:/DataAnalysis/DataAnalysisRoad/Projects/01_Credit_Risk_Analysis/output/tables/risk_segment_summary.csv'),
 WindowsPath('E:/DataAnalysis/DataAnalysisRoad/Projects/01_Credit_Risk_Analysis/output/tables/strategy_impact_high_risk.csv'),
 WindowsPath('E:/DataAnalysis/DataAnalysisRoad/Projects/01_Credit_Risk_Analysis/output/tables/high_risk_reason_summary.csv'),
 WindowsPath('E:/DataAnalysis/DataAnalysisRoad/Projects/01_Credit_Risk_Analysis/output/tables/risk_signal_summary.csv'))

## 10. 本阶段结论

- 本版已明确区分“直接高风险触发信号”“重点叠加信号”“普通叠加信号”和“观察信号”；
- `DPD30+` 和 `开放信贷账户数为0` 都属于重点叠加信号，不能把 `DPD30+` 单独口头标成强信号而把后者降为中等信号；
- 高风险层由直接高风险信号或多个风险信号叠加形成；
- 观察信号由于样本量不足或稳定性不足，不参与正式分层；本版观察信号包括 `家属数量5人以上` 和 `开放信贷账户数大于40`；
- 下一步可以进入 `05_strategy_simulation.ipynb`，比较不同策略阈值下的坏客户捕获和好客户误伤。
